# 청킹 전략별 답변 품질 비교 실험 (Ollama 버전)
- 5주차에서는 chunk_size 300자와 overlap 50자로 고정해서 실습했다. 
- 이번에는 동일한 문서와 동일한 질문을 두고 청킹 파라미터만 바꿔서 답변 품질이 어떻게 변하는지 비교한다.
- **Gemini API 대신 로컬 Ollama(qwen2.5:3b)를 사용한다.**

In [1]:
# 5주차 - parsing.ipynb에서 함수 가져오기
import fitz

def load_pdf(path: str) -> str:
    doc = fitz.open(path)
    return "\n\n".join(page.get_text() for page in doc)

def chunk_text_fixed_overlap(text: str, chunk_size: int = 300, overlap: int = 50):
    """고정 길이로 자르되, 인접 청크가 overlap만큼 겹치도록 한다."""
    assert 0 <= overlap < chunk_size, "overlap은 chunk_size보다 작아야 한다"
    chunks = []
    step = chunk_size - overlap
    for start in range(0, len(text), step):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        if end == len(text):
            break
    return chunks

# 파라미터별 답변 품질 비교

동일한 문서(`part3.pdf`)와 동일한 질문에 대해 `chunk_size`와 `overlap`만 바꿔가며 RAG 답변이 어떻게 달라지는지 비교한다.

In [2]:
import chromadb
import ollama

def ask_ollama(prompt: str) -> str:
    response = ollama.chat(
        model="qwen2.5:3b",
        messages=[{"role": "user", "content": prompt}]
    )
    return response["message"]["content"].strip()

def run_rag(chunk_size: int, overlap: int, query: str) -> tuple[int, str]:
    # 1. 청킹
    text = load_pdf("part3.pdf")
    chunks = chunk_text_fixed_overlap(text, chunk_size=chunk_size, overlap=overlap)

    # 2. 설정별로 분리된 컬렉션에 저장
    client = chromadb.PersistentClient(path="./db")
    col = client.get_or_create_collection(name=f"cmp_{chunk_size}_{overlap}")
    col.upsert(
        ids=[f"c{i}" for i in range(len(chunks))],
        documents=chunks,
        metadatas=[{"source": "part3.pdf"}] * len(chunks)
    )

    # 3. 검색 + 답변 생성
    results = col.query(query_texts=[query], n_results=3)
    context = "\n\n".join(results["documents"][0])
    prompt = (
        "다음 문서를 참고해 답하라.\n"
        "검색 결과에 없는 내용은 모른다고 답하라.\n\n"
        f"[문서]\n{context}\n\n[질문] {query}\n"
    )
    return len(chunks), ask_ollama(prompt)

In [3]:
# 비교할 파라미터 조합
configs = [
    (100,   0),   # 오버랩 없음, 작은 청크
    (100,  20),   # 오버랩 20%, 작은 청크
    (300,   0),   # 오버랩 없음, 중간 청크
    (300,  50),   # 오버랩 17%, 중간 청크
    (500,  50),   # 오버랩 10%, 큰 청크
    (500, 100),   # 오버랩 20%, 큰 청크
]
query = "청킹에서 오버랩이 필요한 이유는 무엇인가요?"

for chunk_size, overlap in configs:
    n_chunks, answer = run_rag(chunk_size, overlap, query)
    print(f"\n{'='*60}")
    print(f"chunk_size={chunk_size:4d}, overlap={overlap:3d}  →  총 {n_chunks}개 청크")
    print(f"{'='*60}")
    print(answer)


chunk_size= 100, overlap=  0  →  총 27개 청크
검색 결과에 없는 내용입니다.

chunk_size= 100, overlap= 20  →  총 34개 청크
청킹에서 오버랩이 필요한 이유는, 고정 길이의 체크를 사용할 때 문장 중간에 쪼개지는 문제가 발생하기 때문입니다. 이는 한 주제가 두 개의 청크로 나눠질 위험을 줄이는 데 도움이 됩니다. 반면, 문단 기반의 체킹은 작성자가 제시한 경계를 따라가므로 한 청 chunk = 한 주제가 되는데, 이는 텍스트에 있는 정보와 질문 사이의 관련성에 대한 검색을 정확하게 수행할 수 있습니다. 또한, 토큰 한도 준수도 중요한 요소 중 하나입니다.

chunk_size= 300, overlap=  0  →  총 9개 청크
검색 결과에 있는 정보를 참조하여 다음과 같이 답할게요.

청킹에서 오버랩이 필요하다는 것은, 한 번에 처리할 수 있는 토큰이나 글자 수의 제약을 극복하고, 질문과 관련된 부분만 정확하게 가져오는 것을 의미합니다. 이렇게 하면 너무 작아서 맥락이 끊긴 청크나, 너무 크게 자른 청크로 인한 무관한 내용이 섞이는 것을 방지할 수 있습니다. 오버랩은 이 문제를 해결하는 중요한 요소입니다.

chunk_size= 300, overlap= 50  →  총 11개 청크
오버랩이 필요한 이유는, 청크 사이를 잇는 맥락을 유지하면서 문장 중간에 겹치도록 자르기 때문에, 무관한 내용이 섞이는 것을 방지하고, 너무 작게 자른 것에서 맥락이 끊어지는 것을 예방하기 위함입니다.

chunk_size= 500, overlap= 50  →  총 6개 청크
청킹에서 오버랩이 필요한 이유는, 문장 중간에서 맥락이 끊기지 않도록 하기 위해서입니다. 이렇게 설정하면 한 번에 처리할 수 있는 텍스트의 길이를 늘릴 수 있으며, 이는 전체 흐름을 유지하면서도 검색 정확도를 향상시킬 수 있습니다.

chunk_size= 500, overlap=100  →  총 7개 청크
오버랩은 문장 중간을 유지하면서 인접한 청크가 

# ChromaDB where 필터 사용법

`query()`에 `where` 인자를 넘기면 메타데이터 조건에 맞는 청크만 임베딩 유사도 검색의 후보가 된다.

지원 연산자: `$eq` · `$ne` · `$gt` · `$gte` · `$lt` · `$lte` · `$in` · `$nin` — 비교와 집합 연산자가 모두 제공된다.

In [4]:
import fitz
import chromadb

client = chromadb.PersistentClient(path="./db")
col = client.get_or_create_collection(name="where_demo")
pdf_files = ["part3.pdf", "part4.pdf"]

for fname in pdf_files:
    doc = fitz.open(fname)
    for i, page in enumerate(doc):
        text = page.get_text().strip()
        if text:
            col.upsert(
                ids=[f"{fname}_page_{i+1}"],
                documents=[text],
                metadatas=[{
                    "source": fname,
                    "page": i + 1,
                    "category": "manual",
                    "lang": "ko",
                    "year": 2025,
                    "n_tokens": len(text) // 4,
                }]
            )

print(f"저장 완료: {col.count()}개 페이지")

# 두 PDF 모두 관련 있는 질문 → 필터 유무에 따라 결과가 달라지는 것을 확인
question = "검색 정확도를 높이려면 어떤 점을 고려해야 하나요?"

filters = {
    "필터 없음":      None,
    "source 필터":    {"source": "part4.pdf"},
    "page 범위 필터": {"page": {"$gte": 2}},
}

for label, where in filters.items():
    results = col.query(query_texts=[question], n_results=3, where=where)
    print(f"\n[{label}]")
    for doc_text, meta in zip(results["documents"][0], results["metadatas"][0]):
        print(f"  page={meta['page']} | {doc_text[:60]}...")

d:\GitHub\1_lecture-materials\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6203.72it/s]


저장 완료: 32개 페이지

[필터 없음]
  page=1 | % 수준에서 시작해 데이터에 맞게 조정
한다.
청킹이 왜 중요한가?
RAG의 검색 정확도는 청크 크기에 크게...
  page=1 | Part 4.  
메타데이터 필터링 — 검색 범위를 좁혀 정확도 높이기
메타데이터는 청크 본문이 아닌 부가 ...
  page=1 | Part 4.  
메타데이터 필터링 — 검색 범위를 좁혀 정확도 높이기
메타데이터는 청크 본문이 아닌 부가 ...

[source 필터]
  page=1 | Part 4.  
메타데이터 필터링 — 검색 범위를 좁혀 정확도 높이기
메타데이터는 청크 본문이 아닌 부가 ...
  page=1 | Part 4.  
메타데이터 필터링 — 검색 범위를 좁혀 정확도 높이기
메타데이터는 청크 본문이 아닌 부가 ...
  page=3 |  돌아온다.
3. where={"source": "part3.pdf"} 를 추가하면 해당 문서의 청크만 결과...

[page 범위 필터]
  page=2 | 경험적으로 한국어 문서는 200500자, 오버랩 1020% 정도가 출발점으로 무난하다.
"정확한 검색은 좋은...
  page=2 | 경험적으로 한국어 문서는 200500자, 오버랩 1020% 정도가 출발점으로 무난하다.
"정확한 검색은 좋은...
  page=3 |  돌아온다.
3. where={"source": "part3.pdf"} 를 추가하면 해당 문서의 청크만 결과...


# K 값을 바꿔 가며 답변 비교

In [8]:
import fitz
import chromadb
import ollama

# Ollama nomic-embed-text를 ChromaDB 임베딩 함수로 래핑
class OllamaEmbeddingFn:
    def __call__(self, input):
        return [
            ollama.embed(model="nomic-embed-text", input=text)["embeddings"][0]
            for text in input
        ]

    def embed_query(self, input):
        # ChromaDB가 query() 시 __call__ 대신 embed_query를 호출함
        return self(input)

def chunk_text_fixed_overlap(text, chunk_size=300, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

client = chromadb.PersistentClient(path="./db")
try:
    client.delete_collection("topk_ol")
except:
    pass

col = client.create_collection(
    name="topk_ol",
    embedding_function=OllamaEmbeddingFn()
)

pdf_files = ["part3.pdf", "part4.pdf"]

for fname in pdf_files:
    doc = fitz.open(fname)
    for i, page in enumerate(doc):
        text = page.get_text().strip()
        if not text:
            continue
        chunks = chunk_text_fixed_overlap(text, chunk_size=300, overlap=50)
        for j, chunk in enumerate(chunks):
            col.add(
                ids=[f"{fname}_p{i+1}_chunk{j}"],
                documents=[chunk],
                metadatas=[{
                    "source": fname,
                    "page": i + 1,
                    "chunk_index": j,
                    "category": "manual",
                    "lang": "ko",
                }]
            )

print(f"저장 완료: {col.count()}개 청크")

questions = [
    "문서를 검색에 적합하게 준비하려면 어떤 과정을 거쳐야 하나요?",
    "검색 정확도를 높이려면 어떤 점을 고려해야 하나요?",
]

for question in questions:
    print(f"\n{'='*70}")
    print(f"질문: {question}")
    print('='*70)
    for k in [1, 3, 5, 10]:
        result = col.query(query_texts=[question], n_results=k)
        print(f"\n--- K={k} (검색된 청크 {len(result['documents'][0])}개) ---")
        for idx, (doc_text, meta) in enumerate(
            zip(result["documents"][0], result["metadatas"][0])
        ):
            preview = doc_text.replace('\n', ' ')[:100]
            print(f"  {idx+1}. [{meta['source']} p{meta['page']}] {preview}...")

저장 완료: 26개 청크

질문: 문서를 검색에 적합하게 준비하려면 어떤 과정을 거쳐야 하나요?

--- K=1 (검색된 청크 1개) ---
  1. [part3.pdf p3] 장한다. 문단 기반 청킹 \n\n 으로 자르되 너무 긴 문단은 추가로 잘라준다. 의미 단위가 잘 보존된다. 글을 쓸 때 사람은 하나의 주제·논점이 끝났을 때만 빈 줄( \n\n )...

--- K=3 (검색된 청크 3개) ---
  1. [part3.pdf p3] 장한다. 문단 기반 청킹 \n\n 으로 자르되 너무 긴 문단은 추가로 잘라준다. 의미 단위가 잘 보존된다. 글을 쓸 때 사람은 하나의 주제·논점이 끝났을 때만 빈 줄( \n\n )...
  2. [part3.pdf p2] 은 의미 단위를 무시하고 글자 수만 보고 자르기 때문에, 청크 경계가 문장 한 가운데 떨어지는 일이 흔하다. 오버랩은 인접 청크가 일부 겹치도록 만들어 그 경계에서 끊 긴 맥락을 ...
  3. [part3.pdf p1] 으면 맥락이 끊긴다. 비용과 속도: LLM에 넘기는 컨텍스트가 짧을수록 응답이 빠르고 토큰 비용도 줄어든다. 어떻게 자르는가 고정 길이 분할: 글자 수나 토큰 수 기준으로 일정하게...

--- K=5 (검색된 청크 5개) ---
  1. [part3.pdf p3] 장한다. 문단 기반 청킹 \n\n 으로 자르되 너무 긴 문단은 추가로 잘라준다. 의미 단위가 잘 보존된다. 글을 쓸 때 사람은 하나의 주제·논점이 끝났을 때만 빈 줄( \n\n )...
  2. [part3.pdf p2] 은 의미 단위를 무시하고 글자 수만 보고 자르기 때문에, 청크 경계가 문장 한 가운데 떨어지는 일이 흔하다. 오버랩은 인접 청크가 일부 겹치도록 만들어 그 경계에서 끊 긴 맥락을 ...
  3. [part3.pdf p1] 으면 맥락이 끊긴다. 비용과 속도: LLM에 넘기는 컨텍스트가 짧을수록 응답이 빠르고 토큰 비용도 줄어든다. 어떻게 자르는가 고정 길이 분할: 글자 수나 토큰 수 기준으로 일정하게...
  4

# 프롬프트 설계

검색된 청크를 LLM이 잘 쓰게 만드는 건 결국 **프롬프트의 구조**다.  
같은 컨텍스트라도 어떻게 묻느냐에 따라 답의 정확도와 형식이 크게 달라진다.

| 비유 | 역할 |
|------|------|
| 재료 | 검색된 청크 (Context) |
| 레시피 | 프롬프트 (Prompt) |

이번 실습에서는 **동일한 문서와 동일한 질문**을 두고 프롬프트 구조만 바꿔 답변이 어떻게 달라지는지 비교한다.

In [9]:
import fitz
import chromadb
import ollama

def ask_ollama(prompt: str) -> str:
    return ollama.chat(
        model="qwen2.5:3b",
        messages=[{"role": "user", "content": prompt}]
    )["message"]["content"].strip()

def load_pdf(path: str) -> str:
    doc = fitz.open(path)
    return "\n\n".join(page.get_text() for page in doc)

def chunk_text_fixed_overlap(text: str, chunk_size: int = 300, overlap: int = 50):
    chunks = []
    step = chunk_size - overlap
    for start in range(0, len(text), step):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        if end == len(text):
            break
    return chunks

client = chromadb.PersistentClient(path="./db")
try:
    client.delete_collection("prompt_demo")
except:
    pass
col = client.create_collection(name="prompt_demo")

text = load_pdf("part3.pdf")
chunks = chunk_text_fixed_overlap(text, chunk_size=300, overlap=50)
col.add(
    ids=[f"c{i}" for i in range(len(chunks))],
    documents=chunks,
    metadatas=[{"source": "part3.pdf"}] * len(chunks)
)
print(f"저장 완료: {col.count()}개 청크")

저장 완료: 11개 청크


## 컨텍스트 검색 — 모든 프롬프트에서 동일하게 사용

검색 결과(재료)는 고정한 채 프롬프트(레시피)만 바꾼다.

In [10]:
query = "청킹에서 오버랩이 필요한 이유는 무엇인가요?"

results = col.query(query_texts=[query], n_results=3)
context = "\n\n".join(results["documents"][0])

print("[검색된 컨텍스트]")
print(context[:300], "...")

[검색된 컨텍스트]
력 질문: "환불 정책은 어떻게 되나요?"
같은 PDF, 같은 질문이라도 청킹 전략에 따라 가져오는 청크가 달라지고 답변 품질도 함께
바뀐다.
첫째, 오버랩이 있는 청킹은 답이 청크 경계에 걸쳐 있어도 안정적으로 찾는다.
둘째, 문단 청킹은 짧은 문단을 잘 보존하지만 긴 문단에서는 정확도가 떨어질 수 있다.
Part 3.
3


Part 3.
청킹 — 문서를 검색 단위로 자르기
청킹은 긴 문서를 검색에 적합한 작은 조각으로 나누는 작업이다. 임베딩 모델과 LLM은 한
번에 처리할 수 있는 길이가 제한되어 있고(컨텍스트 윈도우), 검 ...


## 프롬프트 유형 비교

| # | 유형 | 특징 |
|---|------|------|
| 1 | 구조 없음 | 지시·자료·질문이 뒤섞임 |
| 2 | 기본 구조 | `[문서]` / `[질문]` 태그로 분리 |
| 3 | 역할 + 제약 + 형식 | 역할 부여 · 없는 내용 금지 · 출력 형식 지정 |
| 4 | Chain-of-Thought | 단계별 추론 후 최종 답변 |

In [11]:
prompts = {
    "1. 구조 없음": (
        f"{context} 위 내용을 참고해서 {query}에 대해 답해줘."
    ),
    "2. 기본 구조": (
        f"다음 문서를 참고해 답하라.\n\n[문서]\n{context}\n\n[질문] {query}"
    ),
    "3. 역할 + 제약 + 형식": (
        f"당신은 RAG 시스템의 답변 생성기입니다.\n"
        f"규칙:\n"
        f"- [문서]에 있는 내용만 답하세요.\n"
        f"- 문서에 없는 내용은 '문서에서 확인할 수 없습니다'라고 답하세요.\n"
        f"- 답변은 bullet point(•)로 정리하세요.\n\n"
        f"[문서]\n{context}\n\n[질문] {query}"
    ),
    "4. Chain-of-Thought": (
        f"다음 문서를 읽고 질문에 답하라.\n"
        f"단계별로 생각한 뒤 최종 답변을 작성하라.\n\n"
        f"[문서]\n{context}\n\n[질문] {query}\n\n"
        f"[풀이]\n1단계 — 문서에서 관련 문장 찾기:\n"
        f"2단계 — 핵심 이유 정리:\n3단계 — 최종 답변:"
    ),
}

for label, prompt in prompts.items():
    answer = ask_ollama(prompt)
    print(f"\n{'='*60}")
    print(f"{label}")
    print('='*60)
    print(answer)


1. 구조 없음
청킹에서 오버랩이 필요하다는 주된 이유는, 질문과 관련된 정보를 정확하게 추출하고 싶지만, 전체 문서를 한 번에 처리하기 어렵다는 것입니다. RAG(Retrieval-Augmented Generation) 구현에서는 임베딩 모델이나 LLM을 사용하여 질문과 유사한 부분만 끌어와야 하는데, 이렇게 하려면 일부 내용이 분리되어 버려지기 쉽습니다.

오버랩은 이 문제를 해결하는데 도움이 됩니다. 청chunk가 문장 중간에서 시작하고 종료되는 대신, 이웃하는 청chunk 사이에 일부 내용을 겹쳐 넣음으로써 맥락을 유지할 수 있게 합니다. 이렇게 하면 질문과 관련된 정보가 더 잘 추출될 수 있습니다.

따라서 오버랩은 문장 간의 자연스러운 경계를 보존함으로써, 질문과 관련된 정보를 정확하게 추출하고 싶어하는 RAG 구현에 도움이 됩니다. 이렇게 하면 맥락이 끊기지 않고, 관련성이 있는 정보가 더 잘 유지될 수 있습니다.

2. 기본 구조
청킹에서 오버랩이 필요하다는 이유는 두 가지 주요 점을 이해해 볼 수 있습니다.

첫째, 문장 중간에서 맥락이 끊기는 문제를 해결합니다. 이것은 문장 사이의 연관성이 약화되고 질문에 대한 정확한 답변으로부터 중요한 정보가 제외될 수 있다는 의미입니다. 오버랩을 통해 이 문제를 해결할 수 있습니다.

둘째, 검색 결과를 더 정확하게 만들기 위함입니다. 긴 문서에서는 일부 청크가 맥락과 연결되어 있지 않을 수도 있지만, 이를 통해 그들의 유의미한 부분이 유지됩니다. 이렇게 하면 전체적인 이해도 향상되며 질문에 대한 답변을 더 신중하게 제공할 수 있습니다.

따라서, 오버랩은 청킹 과정에서 중요한 역할을 합니다. 이를 통해 문장 사이의 연결성과 맥락이 잘 유지되는 상태를 유지하면서 검색 결과가 더욱 정확해집니다.

3. 역할 + 제약 + 형식
• 청킹에서 오버랩이 필요하다는 것은, 문장 중간에 맥락이 끊기지 않도록 하여 답변의 연결성을 유지하기 위함입니다.
• 이를 통해 이웃한 청크 사이에서 맥락이 전달되는 효과를 얻을 수

## 문서에 없는 내용을 질문하면?

프롬프트 구조에 따라 **환각(hallucination)** 발생 여부가 달라진다.

In [12]:
out_of_scope_query = "파이썬에서 데코레이터는 어떻게 동작하나요?"

results2 = col.query(query_texts=[out_of_scope_query], n_results=3)
context2 = "\n\n".join(results2["documents"][0])

hallucination_prompts = {
    "제약 없음 (환각 위험)": (
        f"[문서]\n{context2}\n\n[질문] {out_of_scope_query}"
    ),
    "제약 있음 (환각 방지)": (
        f"규칙: [문서]에 있는 내용만 답하고, 없으면 '문서에서 확인할 수 없습니다'라고 답하라.\n\n"
        f"[문서]\n{context2}\n\n[질문] {out_of_scope_query}"
    ),
}

for label, prompt in hallucination_prompts.items():
    answer = ask_ollama(prompt)
    print(f"\n{'='*60}")
    print(f"{label}")
    print('='*60)
    print(answer)


제약 없음 (환각 위험)
I understand that you provided a section of text related to document chunking (or "청킹" in Korean), but your question is about Python decorators. I'll be happy to explain how Python decorators work!

A decorator is a design pattern used in Python to modify or enhance the behavior of functions, classes, modules, or methods without permanently altering their code. It's essentially syntactic sugar over function calls.

Here’s how a basic decorator works:

1. **Definition**: A decorator takes another function as an argument and returns a new function that wraps around the original one.
2. **Wrapper Function**: The returned wrapper function contains the logic that modifies or enhances the behavior of the decorated function.

Let's see it in code:

```python
def my_decorator(func):
    def wrapper():
        print("Something is happening before func is called.")
        func()
        print("Something is happening after func has been called.")
    return wrapper

@my_decorator 

# 개선 전략 ① — 명시적 제약을 추가한다

프롬프트 안에 **"문서만을 근거로"**, **"모르면 모른다고 답하라"** 는 규칙을 명시적으로 적는다.

| | Before (기본 프롬프트) | After (개선 프롬프트) |
|---|---|---|
| 지시 | 아래 문서를 참고하여 질문에 답하세요. | 아래 문서**만을 근거로** 답하세요. |
| 제약 | (없음) | 문서에 없으면 **"찾을 수 없습니다"** 라고 답하세요. |
| 금지 | (없음) | 추측하거나 **지어내지 마세요**. |

In [13]:
import fitz
import chromadb
import ollama

def ask_ollama(prompt: str) -> str:
    return ollama.chat(
        model="qwen2.5:3b",
        messages=[{"role": "user", "content": prompt}]
    )["message"]["content"].strip()

def load_pdf(path: str) -> str:
    doc = fitz.open(path)
    return "\n\n".join(page.get_text() for page in doc)

def chunk_text_fixed_overlap(text: str, chunk_size: int = 300, overlap: int = 50):
    chunks = []
    step = chunk_size - overlap
    for start in range(0, len(text), step):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        if end == len(text):
            break
    return chunks

client = chromadb.PersistentClient(path="./db")
try:
    client.delete_collection("constraint_demo")
except:
    pass
col = client.create_collection(name="constraint_demo")

text = load_pdf("part3.pdf")
chunks = chunk_text_fixed_overlap(text, chunk_size=300, overlap=50)
col.add(
    ids=[f"c{i}" for i in range(len(chunks))],
    documents=chunks,
    metadatas=[{"source": "part3.pdf"}] * len(chunks)
)
print(f"저장 완료: {col.count()}개 청크")

저장 완료: 11개 청크


## 실험 1 — 문서 안에 있는 질문

정상적인 질문에서는 Before/After 모두 비슷하게 답하는지 확인한다.

In [14]:
query = "청킹에서 오버랩이 필요한 이유는 무엇인가요?"

results = col.query(query_texts=[query], n_results=3)
context = "\n\n".join(results["documents"][0])

# ── Before ───────────────────────────────────────────────
before_prompt = (
    "아래 문서를 참고하여\n질문에 답하세요.\n\n"
    f"[검색 결과]\n{context}\n\n질문: {query}"
)

# ── After ────────────────────────────────────────────────
after_prompt = (
    "아래 문서만을 근거로 답하세요.\n"
    "문서에 해당 내용이 없으면 '제공된 문서에서 해당 정보를 찾을 수 없습니다'라고 답하세요.\n"
    "문서에 없는 내용을 추측하거나 지어내지 마세요.\n\n"
    f"[검색 결과]\n{context}\n\n질문: {query}"
)

print(f"질문: {query}\n")

print("=" * 60)
print("[Before] 기본 프롬프트")
print("=" * 60)
print(ask_ollama(before_prompt))

print("\n" + "=" * 60)
print("[After] 개선 프롬프트")
print("=" * 60)
print(ask_ollama(after_prompt))

질문: 청킹에서 오버랩이 필요한 이유는 무엇인가요?

[Before] 기본 프롬프트
청킹에서 오버랩이 필요한 주된 이유는 다음과 같습니다:

1. **맥락 유지**: 문장 간에 의미를 이은 맥락을 유지하려고 하면, 두 개의 청크 사이에는 자연스럽게 연결될 수 있는 부분들이 생깁니다. 이러한 연결점들은 오버랩으로 보완되어 서로 독립적인 세그먼트가 연결되도록 합니다.

2. **관련성 유지**: 검색 결과에서 질문과 관련성이 높은 내용만을 선택하기 위해, 문장들 사이에 자연스러운 맥락이 있을 때 그 맥락을 유지하면 훨씬 더 정확한 답변을 얻을 수 있습니다.

3. **정확도 증가**: 문장을 단일 청크로 분할하는 대신 두 개의 청chunk 사이를 중복해서 잘라내는 방법으로 검색 결과를 구성하게 되면, 오버랩은 관련된 정보들이 서로 연결되어 있는 것을 보장합니다. 이는 특히 긴 문서나 복잡한 내용에서는 더욱 중요합니다.

따라서, 청킹에서 오버랩을 사용하면 문맥이 끊어진 상태에서 서로 관련성이 없는 두 개의 구문 사이에 자연스럽게 맥락이 형성될 수 있습니다. 이는 결국 검색 결과가 질문과 더 잘 일치하며 정확한 답변을 제공할 가능성을 높이는 효과적인 방법입니다.

[After] 개선 프롬프트
청킹에서 오버랩이 필요한 이유는 다음과 같습니다:

1. **맥락 유지**: 문장 중간에 자르면 맥락이 끊기게 되는데, 이를 해결하기 위해 인접한 청크 간에 일부 공통 부분을 포함시키는 "오버랩" 기법을 사용합니다.

2. **정확도 향상**: 이웃하는 청크가 일부 겹치도록 자른다면, 한 문장에서 발생할 수 있는 문제점이나 뜻이 분명하지 않은 부분이 다른 문장에서 해결될 가능성이 높아집니다.

3. **오류 줄이기**: 인공지능 모델의 학습 데이터에 대한 이해가 불충분한 경우에는 한 문장에서 발생할 수 있는 오류를 다른 문장에서 찾아서 보완할 수도 있습니다.

따라서, 오버랩은 맥락을 유지하고 정확도를 높이기 위해 청킹 과정에서 중요하게 다루어집니다.


## 실험 2 — 문서 밖에 있는 질문 (핵심 실험)

part3.pdf에 없는 내용을 질문했을 때,  
- **Before**: LLM이 자체 지식으로 답을 만들어 낼 수 있다 → **환각(Hallucination)**  
- **After**: "찾을 수 없습니다"라고 정직하게 거부한다

In [15]:
out_query = "파이썬에서 데코레이터는 어떻게 동작하나요?"

results2 = col.query(query_texts=[out_query], n_results=3)
context2 = "\n\n".join(results2["documents"][0])

before_prompt2 = (
    "아래 문서를 참고하여\n질문에 답하세요.\n\n"
    f"[검색 결과]\n{context2}\n\n질문: {out_query}"
)

after_prompt2 = (
    "아래 문서만을 근거로 답하세요.\n"
    "문서에 해당 내용이 없으면 '제공된 문서에서 해당 정보를 찾을 수 없습니다'라고 답하세요.\n"
    "문서에 없는 내용을 추측하거나 지어내지 마세요.\n\n"
    f"[검색 결과]\n{context2}\n\n질문: {out_query}"
)

print(f"질문: {out_query}\n")

print("=" * 60)
print("[Before] 기본 프롬프트  →  환각 발생 가능")
print("=" * 60)
print(ask_ollama(before_prompt2))

print("\n" + "=" * 60)
print("[After]  개선 프롬프트  →  정직하게 거부")
print("=" * 60)
print(ask_ollama(after_prompt2))

질문: 파이썬에서 데코레이터는 어떻게 동작하나요?

[Before] 기본 프롬프트  →  환각 발생 가능
제공된 문서에 대한 정보로는, 파이썬에서의 데코레이터와 그 작동 방식을 설명하는 내용은 포함되어 있지 않습니다. 제공된 문서는 주로 텍스트 검색과 자료 분석, 특히 청킹(Chunking) 방법에 대해 다룹니다. 따라서 이 질문에 대한 답변을 드릴 수는 없습니다.

대신 파이썬에서 데코레이터의 기본 원리를 설명하면 다음과 같습니다:

데코레이터는 함수나 메소드를 수정하거나 확장하기 위해 사용되는 특별한 유형의 래퍼입니다. 데코레이터는 실제로 동작하는 코드가 아닌, 다른 함수 또는 메소드에 대해 추가 정보를 제공하게 합니다.

데코레이터의 기본적인 구조와 작동 방식은 다음과 같습니다:

1. 데코레이터는 항상 함수 형태로 정의됩니다.
2. 데코레이터는 두 개 이상의 매개변수를 받을 수 있습니다.
3. 가장 중요한 매개변수인 `func`에 따라 처리됩니다. 이 매개변수는 래퍼가 실제로 수정하고자 하는 원본 함수 또는 메소드입니다.
4. 데코레이터 내부에서, 원본 함수를 호출하여 결과를 생성합니다.
5. 데코레이터의 반환 값은 새 함수를 의미하며, 이 새 함수는 원본 함수의 동작을 그대로 복제합니다.

예시로 간단한 데코레이터 구현을 보겠습니다:

```python
def my_decorator(func):
    def wrapper(*args, **kwargs):
        print("Something is happening before the function is called.")
        result = func(*args, **kwargs)
        print("Something is happening after the function is called.")
        return result
    return wrapper

@my_decorator
def say_hello(name):
    print(f"Hello {name}"

## 개선 전략 ② — 역할 부여와 단계적 사고 유도

3주차에서 배운 시스템 프롬프트와 few-shot 기법을 RAG에 결합한다.  
역할을 부여하고 "인용 → 답변" 순서로 사고하게 만든다.

**시스템 프롬프트에 역할 부여**
```
당신은 문서 기반 QA 전문가입니다.
반드시 제공된 문서만을 근거로 답변하며, 문서에 없는 내용은 "문서에 없음"이라고 답합니다.
```

**Chain-of-Thought 유도**
```
1) 먼저 답변의 근거가 되는 문서 내용을 "인용:" 으로 표시하세요.
2) 그 다음 "답변:" 으로 실제 답변을 작성하세요.
3) 근거를 찾을 수 없으면 인용 없이 "문서에서 찾을 수 없음" 이라고 답하세요.
```

> **역할 + 단계 지시를 결합하면 LLM이 "근거 없이 답하는" 경로 자체가 막힌다.**

In [16]:
col = client.get_collection("constraint_demo")

def make_role_cot_prompt(context: str, query: str) -> str:
    return (
        "당신은 문서 기반 QA 전문가입니다.\n"
        "반드시 제공된 문서만을 근거로 답변하며, 문서에 없는 내용은 \"문서에 없음\"이라고 답합니다.\n\n"
        "1) 먼저 답변의 근거가 되는 문서 내용을 \"인용:\" 으로 표시하세요.\n"
        "2) 그 다음 \"답변:\" 으로 실제 답변을 작성하세요.\n"
        "3) 근거를 찾을 수 없으면 인용 없이 \"문서에서 찾을 수 없음\" 이라고 답하세요.\n\n"
        f"[검색 결과]\n{context}\n\n질문: {query}"
    )

# 문서 안 질문
query_in = "청킹에서 오버랩이 필요한 이유는 무엇인가요?"
results_in = col.query(query_texts=[query_in], n_results=3)
context_in = "\n\n".join(results_in["documents"][0])

# 문서 밖 질문
query_out = "파이썬에서 데코레이터는 어떻게 동작하나요?"
results_out = col.query(query_texts=[query_out], n_results=3)
context_out = "\n\n".join(results_out["documents"][0])

for label, ctx, q in [
    ("문서 안 질문", context_in, query_in),
    ("문서 밖 질문", context_out, query_out),
]:
    prompt = make_role_cot_prompt(ctx, q)
    answer = ask_ollama(prompt)
    print(f"\n{'='*60}")
    print(f"[역할 + CoT] {label}")
    print(f"질문: {q}")
    print('='*60)
    print(answer)


[역할 + CoT] 문서 안 질문
질문: 청킹에서 오버랩이 필요한 이유는 무엇인가요?
문서에서 찾을 수 없음

[역할 + CoT] 문서 밖 질문
질문: 파이썬에서 데코레이터는 어떻게 동작하나요?
문서에서 찾을 수 없음


## 환각 방지 프롬프트 — 네 가지 버전 비교

같은 질문 두 종류(문서에 있는 것 / 없는 것)를 네 가지 프롬프트로 던졌을 때 답변이 어떻게 달라지는지 한눈에 본다.

| 프롬프트 버전 | 문서에 있는 질문 | 문서에 없는 질문 |
|---|---|---|
| **기본** | 답변 OK | 환각 발생 |
| **제약 추가** | 답변 OK | "찾을 수 없습니다" |
| **출처 표기** | 출처와 함께 답변 | 출처 없이 거부 |
| **역할 + CoT** | 인용 후 답변 | 근거 없음을 명시 |

> 세 가지 전략은 배타적이지 않다. 실전에서는 보통 함께 사용한다.

In [17]:
col = client.get_collection("constraint_demo")

def make_prompt(version: str, context: str, query: str) -> str:
    if version == "기본":
        return (
            f"아래 문서를 참고하여 질문에 답하세요.\n\n"
            f"[검색 결과]\n{context}\n\n질문: {query}"
        )
    elif version == "제약 추가":
        return (
            "아래 문서만을 근거로 답하세요.\n"
            "문서에 해당 내용이 없으면 \"제공된 문서에서 해당 정보를 찾을 수 없습니다\"라고 답하세요.\n"
            "문서에 없는 내용을 추측하거나 지어내지 마세요.\n\n"
            f"[검색 결과]\n{context}\n\n질문: {query}"
        )
    elif version == "출처 표기":
        return (
            "아래 문서를 참고해 질문에 답하세요.\n"
            "답변에는 근거가 된 문장을 반드시 [출처]: \"...\" 형태로 먼저 표기하세요.\n"
            "문서에 없는 내용은 출처 없이 \"문서에서 찾을 수 없습니다\"라고 답하세요.\n\n"
            f"[검색 결과]\n{context}\n\n질문: {query}"
        )
    elif version == "역할 + CoT":
        return (
            "당신은 문서 기반 QA 전문가입니다.\n"
            "반드시 제공된 문서만을 근거로 답변하며, 문서에 없는 내용은 \"문서에 없음\"이라고 답합니다.\n\n"
            "1) 먼저 답변의 근거가 되는 문서 내용을 \"인용:\" 으로 표시하세요.\n"
            "2) 그 다음 \"답변:\" 으로 실제 답변을 작성하세요.\n"
            "3) 근거를 찾을 수 없으면 인용 없이 \"문서에서 찾을 수 없음\" 이라고 답하세요.\n\n"
            f"[검색 결과]\n{context}\n\n질문: {query}"
        )

# 두 종류의 질문
q_in  = "청킹에서 오버랩이 필요한 이유는 무엇인가요?"   # 문서 안
q_out = "파이썬에서 데코레이터는 어떻게 동작하나요?"    # 문서 밖

ctx_in  = "\n\n".join(col.query(query_texts=[q_in],  n_results=3)["documents"][0])
ctx_out = "\n\n".join(col.query(query_texts=[q_out], n_results=3)["documents"][0])

versions = ["기본", "제약 추가", "출처 표기", "역할 + CoT"]

for ver in versions:
    print(f"\n{'='*65}")
    print(f"[{ver}]")
    print('='*65)

    for label, ctx, q in [("문서 안 질문", ctx_in, q_in), ("문서 밖 질문", ctx_out, q_out)]:
        prompt = make_prompt(ver, ctx, q)
        answer = ask_ollama(prompt)
        print(f"\n  ▶ {label}: {q}")
        print(f"  {answer[:200]}{'...' if len(answer) > 200 else ''}")


[기본]

  ▶ 문서 안 질문: 청킹에서 오버랩이 필요한 이유는 무엇인가요?
  청킹에서 오버랩이 필요한 주된 이유는 다음과 같습니다:

1. 맥락 보존: 문서의 각 부분을 잘 분리했지만, 문장과 문장 사이에는 지루한 허물은 없습니다. 이를 해결하기 위해 오버랩이 사용됩니다. 이로써 청크 간의 맥락적 연관성이 유지되며, 전반적인 내용 이해를 돕습니다.

2. 안정적인 응답: 답변이 여러 청크 사이에 걸쳐 있어도, 오버랩은 답이 청크 경...

  ▶ 문서 밖 질문: 파이썬에서 데코레이터는 어떻게 동작하나요?
  파이썬에서 디코더(Decorator)는 함수와 클래스를 감싸서 새로운 속성을 추가하는 기법입니다. 그들은 원래의 코드를 변경하지 않고도 별도의 로직을 수행하고, 특정 메서드가 호출될 때마다 실행되는 동작이 가능합니다.

디코더의 기본 동작은 다음과 같습니다:

1. **Decorated Function**: 디코더는 이미 존재하는 기능(function)에 ...

[제약 추가]

  ▶ 문서 안 질문: 청킹에서 오버랩이 필요한 이유는 무엇인가요?
  청킹에서 오버랩이 필요한 주된 이유는 다음과 같습니다:

1. **맥락 유지**: 문장 사이에 자연스러운 경계를 두고 분할하면, 그 사이의 맥락을 잃어버릴 수 있습니다. 이로 인해 관련성이 떨어지는 부분이 포함될 수 있어 검색 결과가 정확하지 않을 수 있기 때문입니다.

2. **관련성 유지**: 청크에서 오버랩은 주제와 관련된 부분만 선택하여 맥락을 잃지...

  ▶ 문서 밖 질문: 파이썬에서 데코레이터는 어떻게 동작하나요?
  제공된 문서에서 해당 정보를 찾을 수 없습니다.

[출처 표기]

  ▶ 문서 안 질문: 청킹에서 오버랩이 필요한 이유는 무엇인가요?
  [출처]: 청킹에서 오버랩이 필요한 이유는 문장 중간에 맥락을 끊지 않도록 하기 위함입니다. 이웃한 청크가 일부 겹치게 잘라진 덕분에, 질문과 관련된 부분만 정확하게 추출하고, 그 외의 무관한 내용이 섞임을 줄일 수 있습니다. 문장 맨 앞이